# Final Full Architecture Notebook (Tokenizer + Embeddings + Attention + Generation Eval)

This notebook is a single end-to-end LLM training system.

Pipeline stages in one place:
- Stage 1: Byte-level BPE tokenizer train/load
- Stage 2: SGNS embedding train/load
- Stage 3: Transformer attention language model train
- Stage 4: Generation evaluation + artifact export

Attention variants included and tested:
- Type 1: Multi-Head Self-Attention (MHA)
- Type 2: Masked Causal Self-Attention
- Type 3: Multi-Query Attention (MQA)
- Type 4: Grouped-Query Attention (GQA)
- Bonus: Cross-Attention

Core architecture quality features:
- RMSNorm pre-norm blocks
- SwiGLU feed-forward network
- RoPE positional encoding
- weight tying
- CPU + RTX 4060 profile scaling

Tip: set `FORCE_RETRAIN_TOKENIZER` and `FORCE_RETRAIN_EMBEDDINGS` below to `True` if you want to retrain all stages from scratch in this notebook run.

In [1]:
from __future__ import annotations

import json
import math
import random
import re
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")

Device: cpu


In [2]:
@dataclass
class TokenizerConfig:
    vocab_size: int = 2000
    min_pair_freq: int = 2
    special_tokens: Tuple[str, ...] = ("<pad>", "<bos>", "<eos>", "<unk>")


class BPETokenizerRuntime:
    """Byte-level BPE tokenizer with train/load/save support."""

    _word_re = re.compile(r"\s+|[^\s]+")

    def __init__(self, config: TokenizerConfig | None = None):
        self.config = config or TokenizerConfig()
        self.base_vocab_size = 256
        self.special_tokens = list(self.config.special_tokens)

        self.special_to_id: Dict[str, int] = {}
        self.id_to_special: Dict[int, str] = {}
        self.merges: Dict[Tuple[int, int], int] = {}
        self.merges_rank: Dict[Tuple[int, int], int] = {}
        self.token_to_bytes: Dict[int, bytes] = {i: bytes([i]) for i in range(self.base_vocab_size)}
        self._init_special_tokens()

    @property
    def vocab_size(self) -> int:
        return len(self.token_to_bytes)

    def _init_special_tokens(self) -> None:
        start = self.base_vocab_size
        for i, tok in enumerate(self.special_tokens):
            tid = start + i
            self.special_to_id[tok] = tid
            self.id_to_special[tid] = tok
            self.token_to_bytes[tid] = tok.encode("utf-8")

    def _merge_sequence(self, seq: Tuple[int, ...], pair: Tuple[int, int], new_id: int) -> Tuple[int, ...]:
        if len(seq) < 2:
            return seq

        out = []
        i = 0
        a, b = pair
        n = len(seq)
        while i < n:
            if i < n - 1 and seq[i] == a and seq[i + 1] == b:
                out.append(new_id)
                i += 2
            else:
                out.append(seq[i])
                i += 1
        return tuple(out)

    def train(self, text: str, verbose: bool = True) -> None:
        if not text:
            raise ValueError("Cannot train tokenizer on empty text.")

        chunks = self._word_re.findall(text)
        word_freqs = Counter(tuple(chunk.encode("utf-8")) for chunk in chunks)

        max_merges = self.config.vocab_size - (self.base_vocab_size + len(self.special_tokens))
        if max_merges <= 0:
            raise ValueError("vocab_size is too small for base bytes and special tokens.")

        self.merges.clear()
        self.merges_rank.clear()
        self.token_to_bytes = {i: bytes([i]) for i in range(self.base_vocab_size)}
        self.special_to_id.clear()
        self.id_to_special.clear()
        self._init_special_tokens()

        next_token_id = self.base_vocab_size + len(self.special_tokens)
        start_time = time.perf_counter()
        merge_count = 0

        for merge_step in range(max_merges):
            pair_counts: Counter[Tuple[int, int]] = Counter()
            for symbols, freq in word_freqs.items():
                for i in range(len(symbols) - 1):
                    pair_counts[(symbols[i], symbols[i + 1])] += freq

            if not pair_counts:
                break

            best_pair, best_freq = pair_counts.most_common(1)[0]
            if best_freq < self.config.min_pair_freq:
                break

            new_id = next_token_id
            next_token_id += 1

            self.merges[best_pair] = new_id
            self.merges_rank[best_pair] = merge_step
            self.token_to_bytes[new_id] = self.token_to_bytes[best_pair[0]] + self.token_to_bytes[best_pair[1]]

            updated = Counter()
            for symbols, freq in word_freqs.items():
                merged = self._merge_sequence(symbols, best_pair, new_id)
                updated[merged] += freq
            word_freqs = updated

            merge_count += 1
            if verbose and (merge_step + 1) % 200 == 0:
                print(f"Tokenizer merges: {merge_step + 1}/{max_merges}")

        if verbose:
            elapsed = time.perf_counter() - start_time
            print(f"Tokenizer trained | merges={merge_count} | vocab={self.vocab_size} | time={elapsed:.2f}s")

    def _encode_chunk(self, chunk: str) -> List[int]:
        symbols: List[int] = list(chunk.encode("utf-8"))
        if len(symbols) < 2:
            return symbols

        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i + 1])
                rank = self.merges_rank.get(pair)
                if rank is not None and rank < best_rank:
                    best_rank = rank
                    best_pair = pair

            if best_pair is None:
                break

            merged_token = self.merges[best_pair]
            out = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best_pair:
                    out.append(merged_token)
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            symbols = out

        return symbols

    def encode(self, text: str, add_bos: bool = False, add_eos: bool = False) -> List[int]:
        if text == "":
            return []

        token_ids: List[int] = []
        if add_bos and "<bos>" in self.special_to_id:
            token_ids.append(self.special_to_id["<bos>"])

        for chunk in self._word_re.findall(text):
            token_ids.extend(self._encode_chunk(chunk))

        if add_eos and "<eos>" in self.special_to_id:
            token_ids.append(self.special_to_id["<eos>"])

        return token_ids

    def decode(self, token_ids: List[int], skip_special_tokens: bool = True) -> str:
        byte_stream = bytearray()
        for tid in token_ids:
            if skip_special_tokens and tid in self.id_to_special:
                continue
            token_bytes = self.token_to_bytes.get(int(tid), b"")
            byte_stream.extend(token_bytes)
        return bytes(byte_stream).decode("utf-8", errors="replace")

    def save(self, path: str | Path) -> None:
        payload = {
            "config": {
                "vocab_size": self.config.vocab_size,
                "min_pair_freq": self.config.min_pair_freq,
                "special_tokens": list(self.special_tokens),
            },
            "merges": [[a, b, new_id] for (a, b), new_id in self.merges.items()],
        }
        Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path: str | Path) -> "BPETokenizerRuntime":
        payload = json.loads(Path(path).read_text(encoding="utf-8"))
        conf = payload["config"]

        tokenizer = cls(
            TokenizerConfig(
                vocab_size=int(conf["vocab_size"]),
                min_pair_freq=int(conf.get("min_pair_freq", 2)),
                special_tokens=tuple(conf.get("special_tokens", ["<pad>", "<bos>", "<eos>", "<unk>"])),
            )
        )

        tokenizer.merges.clear()
        tokenizer.merges_rank.clear()
        tokenizer.token_to_bytes = {i: bytes([i]) for i in range(tokenizer.base_vocab_size)}
        tokenizer.special_to_id.clear()
        tokenizer.id_to_special.clear()
        tokenizer._init_special_tokens()

        for rank, (a, b, new_id) in enumerate(payload["merges"]):
            pair = (int(a), int(b))
            new_id = int(new_id)
            tokenizer.merges[pair] = new_id
            tokenizer.merges_rank[pair] = rank
            tokenizer.token_to_bytes[new_id] = tokenizer.token_to_bytes[pair[0]] + tokenizer.token_to_bytes[pair[1]]

        return tokenizer

In [3]:
project_root = Path("..")
data_path = project_root / "wizard_of_oz.txt"
tokenizer_path = Path("bpe_tokenizer_wizard.json")
embedding_artifact_path = Path("embedding_sgns_wizard.pt")

# Toggle these to retrain complete upstream stages inside this notebook run.
FORCE_RETRAIN_TOKENIZER = False
FORCE_RETRAIN_EMBEDDINGS = False

if not data_path.exists():
    raise FileNotFoundError("wizard_of_oz.txt not found in project root.")

raw_text = data_path.read_text(encoding="utf-8")

if tokenizer_path.exists() and not FORCE_RETRAIN_TOKENIZER:
    tokenizer = BPETokenizerRuntime.load(tokenizer_path)
    print("Loaded tokenizer artifact:", tokenizer_path)
else:
    tokenizer = BPETokenizerRuntime(
        TokenizerConfig(vocab_size=2000, min_pair_freq=2, special_tokens=("<pad>", "<bos>", "<eos>", "<unk>"))
    )
    tokenizer.train(raw_text, verbose=True)
    tokenizer.save(tokenizer_path)
    print("Trained and saved tokenizer:", tokenizer_path)

token_ids = tokenizer.encode(raw_text, add_bos=True, add_eos=True)
vocab_size = tokenizer.vocab_size

split_idx = int(0.9 * len(token_ids))
train_tokens = torch.tensor(token_ids[:split_idx], dtype=torch.long)
val_tokens = torch.tensor(token_ids[split_idx:], dtype=torch.long)

print(f"Corpus chars: {len(raw_text):,}")
print(f"Total token ids: {len(token_ids):,}")
print(f"Vocab size: {vocab_size:,}")
print(f"Train tokens: {len(train_tokens):,} | Val tokens: {len(val_tokens):,}")

Loaded tokenizer artifact: bpe_tokenizer_wizard.json
Corpus chars: 232,309
Total token ids: 102,130
Vocab size: 2,000
Train tokens: 91,917 | Val tokens: 10,213


In [4]:
@dataclass
class EmbeddingTrainConfig:
    dim: int
    window_size: int
    negatives: int
    batch_size: int
    epochs: int
    lr: float
    max_pairs: int
    grad_accum_steps: int = 1


def build_embedding_profiles() -> Dict[str, EmbeddingTrainConfig]:
    return {
        "cpu_safe": EmbeddingTrainConfig(dim=128, window_size=4, negatives=5, batch_size=256, epochs=2, lr=2e-3, max_pairs=220_000),
        "cpu_quality": EmbeddingTrainConfig(dim=192, window_size=5, negatives=6, batch_size=256, epochs=3, lr=1.5e-3, max_pairs=320_000),
        "rtx_4060_balanced": EmbeddingTrainConfig(dim=256, window_size=5, negatives=8, batch_size=1024, epochs=3, lr=2e-3, max_pairs=900_000),
        "rtx_4060_quality": EmbeddingTrainConfig(dim=384, window_size=6, negatives=10, batch_size=1024, epochs=4, lr=1.5e-3, max_pairs=1_400_000),
    }


def build_skipgram_pairs(
    ids: List[int],
    window_size: int,
    max_pairs: int,
    seed: int = 42,
    skip_token_ids: set[int] | None = None,
    ) -> List[Tuple[int, int]]:
    rng = random.Random(seed)
    skip_token_ids = skip_token_ids or set()

    pairs: List[Tuple[int, int]] = []
    n = len(ids)
    for i, center in enumerate(ids):
        if center in skip_token_ids:
            continue

        w = rng.randint(1, window_size)
        left = max(0, i - w)
        right = min(n, i + w + 1)
        contexts = [ids[j] for j in range(left, right) if j != i and ids[j] not in skip_token_ids]
        if not contexts:
            continue

        pairs.append((center, rng.choice(contexts)))
        if len(pairs) >= max_pairs:
            break

    return pairs


class SkipGramPairsDataset(Dataset):
    def __init__(self, pairs: List[Tuple[int, int]]):
        self.centers = torch.tensor([c for c, _ in pairs], dtype=torch.long)
        self.contexts = torch.tensor([ctx for _, ctx in pairs], dtype=torch.long)

    def __len__(self) -> int:
        return len(self.centers)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.centers[idx], self.contexts[idx]


def build_noise_distribution(ids: List[int], vocab_size: int, power: float = 0.75) -> torch.Tensor:
    freqs = np.zeros(vocab_size, dtype=np.float64)
    for tid, count in Counter(ids).items():
        if 0 <= tid < vocab_size:
            freqs[tid] = count

    freqs = np.maximum(freqs, 1e-12)
    probs = freqs ** power
    probs /= probs.sum()
    return torch.tensor(probs, dtype=torch.float32)


class SkipGramNS(nn.Module):
    def __init__(self, vocab_size: int, dim: int):
        super().__init__()
        self.input_embeddings = nn.Embedding(vocab_size, dim)
        self.output_embeddings = nn.Embedding(vocab_size, dim)

        init_range = 0.5 / dim
        nn.init.uniform_(self.input_embeddings.weight, -init_range, init_range)
        nn.init.zeros_(self.output_embeddings.weight)

    def forward(self, centers: torch.Tensor, positive_contexts: torch.Tensor, negative_contexts: torch.Tensor) -> torch.Tensor:
        center_vecs = self.input_embeddings(centers)
        pos_vecs = self.output_embeddings(positive_contexts)
        neg_vecs = self.output_embeddings(negative_contexts)

        pos_scores = torch.sum(center_vecs * pos_vecs, dim=1)
        pos_loss = F.logsigmoid(pos_scores)

        neg_scores = torch.bmm(neg_vecs, center_vecs.unsqueeze(2)).squeeze(2)
        neg_loss = F.logsigmoid(-neg_scores).sum(dim=1)

        return -(pos_loss + neg_loss).mean()


def train_sgns(
    model: SkipGramNS,
    loader: DataLoader,
    noise_dist: torch.Tensor,
    cfg: EmbeddingTrainConfig,
    device: torch.device,
    ) -> List[float]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    model.train()
    noise_dist = noise_dist.to(device)
    history: List[float] = []

    for epoch in range(cfg.epochs):
        running_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, (centers, contexts) in enumerate(loader, start=1):
            centers = centers.to(device)
            contexts = contexts.to(device)
            negatives = torch.multinomial(
                noise_dist,
                centers.size(0) * cfg.negatives,
                replacement=True,
            ).view(centers.size(0), cfg.negatives)

            if device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    loss = model(centers, contexts, negatives) / cfg.grad_accum_steps
                scaler.scale(loss).backward()
            else:
                loss = model(centers, contexts, negatives) / cfg.grad_accum_steps
                loss.backward()

            should_step = (step % cfg.grad_accum_steps == 0) or (step == len(loader))
            if should_step:
                if device.type == "cuda":
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item() * cfg.grad_accum_steps

        epoch_loss = running_loss / max(1, len(loader))
        history.append(epoch_loss)
        print(f"Embedding epoch {epoch + 1}/{cfg.epochs} | avg_loss={epoch_loss:.4f}")

    return history

In [5]:
embedding_profiles = build_embedding_profiles()
if device.type == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    embedding_profile_name = "rtx_4060_quality" if vram_gb >= 7.5 else "rtx_4060_balanced"
else:
    embedding_profile_name = "cpu_safe"

embedding_cfg = embedding_profiles[embedding_profile_name]
pretrained_token_embedding: Optional[torch.Tensor] = None

if embedding_artifact_path.exists() and not FORCE_RETRAIN_EMBEDDINGS:
    emb_payload = torch.load(embedding_artifact_path, map_location="cpu")
    pretrained_token_embedding = emb_payload.get("token_embedding")
    print("Loaded embedding artifact:", embedding_artifact_path)
else:
    skip_ids = {tokenizer.special_to_id["<pad>"]} if "<pad>" in tokenizer.special_to_id else set()
    pairs = build_skipgram_pairs(
        token_ids,
        window_size=embedding_cfg.window_size,
        max_pairs=embedding_cfg.max_pairs,
        seed=SEED,
        skip_token_ids=skip_ids,
    )

    dataset = SkipGramPairsDataset(pairs)
    loader = DataLoader(
        dataset,
        batch_size=embedding_cfg.batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

    noise_dist = build_noise_distribution(token_ids, vocab_size=vocab_size)
    sgns_model = SkipGramNS(vocab_size=vocab_size, dim=embedding_cfg.dim).to(device)

    print("Embedding profile:", embedding_profile_name)
    print(f"Embedding pairs: {len(pairs):,} | batches/epoch: {len(loader):,}")

    emb_loss_history = train_sgns(sgns_model, loader, noise_dist, embedding_cfg, device)
    pretrained_token_embedding = sgns_model.input_embeddings.weight.detach().cpu()

    torch.save(
        {
            "token_embedding": pretrained_token_embedding,
            "output_embedding": sgns_model.output_embeddings.weight.detach().cpu(),
            "loss_history": emb_loss_history,
            "profile": embedding_profile_name,
            "config": embedding_cfg.__dict__,
            "vocab_size": vocab_size,
            "tokenizer_json": str(tokenizer_path),
        },
        embedding_artifact_path,
    )
    print("Saved embedding artifact:", embedding_artifact_path)

if pretrained_token_embedding is None:
    raise RuntimeError("Failed to prepare token embeddings for attention stage.")
print("Embedding stage ready | shape:", tuple(pretrained_token_embedding.shape))

Loaded embedding artifact: embedding_sgns_wizard.pt
Embedding stage ready | shape: (2000, 128)


In [14]:
@dataclass
class AttentionModelConfig:
    vocab_size: int
    d_model: int
    n_layers: int
    n_heads: int
    n_kv_heads: int
    dropout: float
    max_seq_len: int
    ffn_mult: float = 3.5
    use_rope: bool = True
    tie_weights: bool = True
    attn_variant: str = "gqa"  # mha | causal_mha | mqa | gqa


@dataclass
class AttentionTrainConfig:
    batch_size: int
    lr: float
    weight_decay: float
    steps: int
    warmup_steps: int
    eval_interval: int
    eval_iters: int
    grad_accum_steps: int
    clip_grad: float
    max_new_tokens: int


def build_profiles(vocab_size: int) -> Dict[str, Tuple[AttentionModelConfig, AttentionTrainConfig]]:
    return {
        "cpu_safe": (
            AttentionModelConfig(
                vocab_size=vocab_size, d_model=192, n_layers=4, n_heads=6, n_kv_heads=6,
                dropout=0.1, max_seq_len=128, use_rope=True, attn_variant="causal_mha",
            ),
            AttentionTrainConfig(
                batch_size=24, lr=3e-4, weight_decay=0.1, steps=45, warmup_steps=8,
                eval_interval=15, eval_iters=8, grad_accum_steps=1, clip_grad=1.0, max_new_tokens=120,
            ),
        ),
        "cpu_quality": (
            AttentionModelConfig(
                vocab_size=vocab_size, d_model=256, n_layers=6, n_heads=8, n_kv_heads=4,
                dropout=0.1, max_seq_len=160, use_rope=True, attn_variant="gqa",
            ),
            AttentionTrainConfig(
                batch_size=16, lr=2.5e-4, weight_decay=0.1, steps=100, warmup_steps=16,
                eval_interval=20, eval_iters=10, grad_accum_steps=1, clip_grad=1.0, max_new_tokens=140,
            ),
        ),
        "rtx_4060_balanced": (
            AttentionModelConfig(
                vocab_size=vocab_size, d_model=512, n_layers=8, n_heads=8, n_kv_heads=4,
                dropout=0.1, max_seq_len=256, use_rope=True, attn_variant="gqa",
            ),
            AttentionTrainConfig(
                batch_size=32, lr=3e-4, weight_decay=0.1, steps=400, warmup_steps=40,
                eval_interval=50, eval_iters=20, grad_accum_steps=1, clip_grad=1.0, max_new_tokens=180,
            ),
        ),
        "rtx_4060_quality": (
            AttentionModelConfig(
                vocab_size=vocab_size, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
                dropout=0.1, max_seq_len=384, use_rope=True, attn_variant="gqa",
            ),
            AttentionTrainConfig(
                batch_size=20, lr=2.5e-4, weight_decay=0.1, steps=800, warmup_steps=80,
                eval_interval=80, eval_iters=24, grad_accum_steps=1, clip_grad=1.0, max_new_tokens=220,
            ),
        ),
        "rtx_4060_max": (
            AttentionModelConfig(
                vocab_size=vocab_size, d_model=1024, n_layers=16, n_heads=16, n_kv_heads=4,
                dropout=0.1, max_seq_len=512, use_rope=True, attn_variant="gqa",
            ),
            AttentionTrainConfig(
                batch_size=10, lr=2e-4, weight_decay=0.1, steps=1200, warmup_steps=120,
                eval_interval=100, eval_iters=24, grad_accum_steps=2, clip_grad=1.0, max_new_tokens=260,
            ),
        ),
    }


profiles = build_profiles(vocab_size)
gpu_vram_gb = 0.0
if device.type == "cuda":
    gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

if device.type == "cuda" and gpu_vram_gb >= 7.5:
    default_stage_plan = ["rtx_4060_balanced", "rtx_4060_quality"]
elif device.type == "cuda":
    default_stage_plan = ["rtx_4060_balanced"]
else:
    default_stage_plan = ["cpu_safe", "cpu_quality"]

# Flexible curriculum controls: small stage first, larger stage second.
ATTENTION_STAGE_PLAN = default_stage_plan
STAGE_STEP_SCALE = [0.5, 1.0]
CHECKPOINT_DIR = Path("checkpoints_full_arch")
CHECKPOINT_FILE = CHECKPOINT_DIR / "full_arch_last.pt"
CHECKPOINT_EVERY = 20
RESUME_FROM_CHECKPOINT = True

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

smoke_profile = ATTENTION_STAGE_PLAN[0]
model_cfg, train_cfg = profiles[smoke_profile]
print("Attention stage plan:", ATTENTION_STAGE_PLAN)
print("Step scales:", STAGE_STEP_SCALE)
print("Checkpoint file:", CHECKPOINT_FILE)
print("Smoke-test profile:", smoke_profile)
print("Model config:", model_cfg)
print("Train config:", train_cfg)

Attention stage plan: ['cpu_safe', 'cpu_quality']
Step scales: [0.5, 1.0]
Checkpoint file: checkpoints_full_arch\full_arch_last.pt
Smoke-test profile: cpu_safe
Model config: AttentionModelConfig(vocab_size=2000, d_model=192, n_layers=4, n_heads=6, n_kv_heads=6, dropout=0.1, max_seq_len=128, ffn_mult=3.5, use_rope=True, tie_weights=True, attn_variant='causal_mha')
Train config: AttentionTrainConfig(batch_size=24, lr=0.0003, weight_decay=0.1, steps=45, warmup_steps=8, eval_interval=15, eval_iters=8, grad_accum_steps=1, clip_grad=1.0, max_new_tokens=120)


In [7]:
def adapt_pretrained_embedding(weight: torch.Tensor, target_dim: int, seed: int = 42) -> torch.Tensor:
    if weight.ndim != 2:
        raise ValueError("Expected embedding weight with shape [vocab_size, dim].")

    src_vocab, src_dim = weight.shape
    if src_vocab != vocab_size:
        raise ValueError(f"Embedding vocab mismatch: {src_vocab} vs tokenizer vocab {vocab_size}.")

    if src_dim == target_dim:
        return weight.float()

    if src_dim > target_dim:
        return weight[:, :target_dim].float()

    rng = torch.Generator().manual_seed(seed)
    pad = torch.randn(src_vocab, target_dim - src_dim, generator=rng) * (0.02 / math.sqrt(target_dim))
    return torch.cat([weight.float(), pad], dim=1)


def get_batch(
    split: str,
    batch_size: int,
    seq_len: int,
    device: torch.device,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
    data = train_tokens if split == "train" else val_tokens
    if len(data) <= seq_len + 1:
        raise ValueError("Sequence length is larger than available token buffer.")

    idx = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([data[i : i + seq_len] for i in idx])
    y = torch.stack([data[i + 1 : i + seq_len + 1] for i in idx])
    return x.to(device), y.to(device)


def cosine_lr(step: int, total_steps: int, warmup_steps: int, base_lr: float, min_lr_ratio: float = 0.1) -> float:
    if step < warmup_steps:
        return base_lr * (step + 1) / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return base_lr * (min_lr_ratio + (1 - min_lr_ratio) * cosine)

In [8]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.scale * x * rms


class SwiGLU(nn.Module):
    def __init__(self, dim: int, ffn_mult: float = 3.5, dropout: float = 0.0):
        super().__init__()
        hidden = int(ffn_mult * dim)
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w_out = nn.Linear(hidden, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.silu(self.w1(x)) * self.w2(x)
        x = self.w_out(x)
        return self.dropout(x)


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, base: float = 10000.0):
        super().__init__()
        if head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE.")
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def get_cos_sin(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> Tuple[torch.Tensor, torch.Tensor]:
        t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos().to(dtype=dtype).unsqueeze(0).unsqueeze(0)
        sin = emb.sin().to(dtype=dtype).unsqueeze(0).unsqueeze(0)
        return cos, sin


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    return (x * cos) + (rotate_half(x) * sin)


class FlexibleAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        n_kv_heads: Optional[int] = None,
        dropout: float = 0.0,
        causal: bool = False,
        use_rope: bool = True,
    ):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads or n_heads
        self.causal = causal
        self.dropout = dropout

        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")
        if n_heads % self.n_kv_heads != 0:
            raise ValueError("n_heads must be divisible by n_kv_heads for grouped KV sharing.")

        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(n_heads * self.head_dim, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)

        self.rope = RotaryEmbedding(self.head_dim) if use_rope else None

    def _repeat_kv(self, x: torch.Tensor) -> torch.Tensor:
        if self.n_kv_heads == self.n_heads:
            return x
        repeat_factor = self.n_heads // self.n_kv_heads
        return x.repeat_interleave(repeat_factor, dim=1)

    def forward(
        self,
        x: torch.Tensor,
        context: Optional[torch.Tensor] = None,
        attn_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        context = x if context is None else context
        bsz, tgt_len, _ = x.shape
        src_len = context.shape[1]

        q = self.q_proj(x).view(bsz, tgt_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(context).view(bsz, src_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(context).view(bsz, src_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if self.rope is not None and context is x:
            cos_q, sin_q = self.rope.get_cos_sin(tgt_len, x.device, q.dtype)
            q = apply_rope(q, cos_q, sin_q)
            cos_k, sin_k = self.rope.get_cos_sin(src_len, x.device, k.dtype)
            k = apply_rope(k, cos_k, sin_k)

        k = self._repeat_kv(k)
        v = self._repeat_kv(v)

        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=self.causal and (attn_mask is None) and (context is x),
        )

        out = attn_out.transpose(1, 2).contiguous().view(bsz, tgt_len, self.d_model)
        return self.o_proj(self.attn_drop(out))


class MultiHeadSelfAttention(FlexibleAttention):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0, use_rope: bool = True):
        super().__init__(d_model=d_model, n_heads=n_heads, n_kv_heads=n_heads, dropout=dropout, causal=False, use_rope=use_rope)


class CausalSelfAttention(FlexibleAttention):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0, use_rope: bool = True):
        super().__init__(d_model=d_model, n_heads=n_heads, n_kv_heads=n_heads, dropout=dropout, causal=True, use_rope=use_rope)


class MultiQueryAttention(FlexibleAttention):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0, causal: bool = True, use_rope: bool = True):
        super().__init__(d_model=d_model, n_heads=n_heads, n_kv_heads=1, dropout=dropout, causal=causal, use_rope=use_rope)


class GroupedQueryAttention(FlexibleAttention):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, dropout: float = 0.0, causal: bool = True, use_rope: bool = True):
        super().__init__(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads, dropout=dropout, causal=causal, use_rope=use_rope)


class CrossAttention(FlexibleAttention):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: Optional[int] = None, dropout: float = 0.0, use_rope: bool = False):
        super().__init__(d_model=d_model, n_heads=n_heads, n_kv_heads=n_kv_heads or n_heads, dropout=dropout, causal=False, use_rope=use_rope)

    def forward(self, x: torch.Tensor, context: torch.Tensor, attn_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        return super().forward(x=x, context=context, attn_mask=attn_mask)

In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: AttentionModelConfig):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.norm2 = RMSNorm(cfg.d_model)

        if cfg.attn_variant == "mha":
            self.attn = MultiHeadSelfAttention(cfg.d_model, cfg.n_heads, dropout=cfg.dropout, use_rope=cfg.use_rope)
        elif cfg.attn_variant == "causal_mha":
            self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, dropout=cfg.dropout, use_rope=cfg.use_rope)
        elif cfg.attn_variant == "mqa":
            self.attn = MultiQueryAttention(cfg.d_model, cfg.n_heads, dropout=cfg.dropout, causal=True, use_rope=cfg.use_rope)
        elif cfg.attn_variant == "gqa":
            self.attn = GroupedQueryAttention(
                cfg.d_model, cfg.n_heads, cfg.n_kv_heads, dropout=cfg.dropout, causal=True, use_rope=cfg.use_rope
            )
        else:
            raise ValueError(f"Unknown attention variant: {cfg.attn_variant}")

        self.ffn = SwiGLU(cfg.d_model, ffn_mult=cfg.ffn_mult, dropout=cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class PowerfulAttentionLM(nn.Module):
    def __init__(self, cfg: AttentionModelConfig, pretrained_embedding: Optional[torch.Tensor] = None):
        super().__init__()
        self.cfg = cfg
        self.token_embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        if cfg.tie_weights:
            self.lm_head.weight = self.token_embed.weight

        self.pos_embed = None if cfg.use_rope else nn.Embedding(cfg.max_seq_len, cfg.d_model)

        if pretrained_embedding is not None:
            init_weight = adapt_pretrained_embedding(pretrained_embedding, cfg.d_model)
            self.token_embed.weight.data.copy_(init_weight)

    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        bsz, seq_len = idx.shape
        if seq_len > self.cfg.max_seq_len:
            raise ValueError(f"Sequence length {seq_len} exceeds max_seq_len {self.cfg.max_seq_len}.")

        x = self.token_embed(idx)
        if self.pos_embed is not None:
            pos = torch.arange(seq_len, device=idx.device).unsqueeze(0).expand(bsz, seq_len)
            x = x + self.pos_embed(pos)

        x = self.dropout(x)
        for block in self.blocks:
            x = block(x)

        x = self.norm_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = 50,
    ) -> torch.Tensor:
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.max_seq_len :]
            logits, _ = self(idx_cond)
            next_logits = logits[:, -1, :] / max(temperature, 1e-6)

            if top_k is not None:
                top_vals, top_idx = torch.topk(next_logits, k=min(top_k, next_logits.size(-1)), dim=-1)
                filtered = torch.full_like(next_logits, float("-inf"))
                filtered.scatter_(1, top_idx, top_vals)
                next_logits = filtered

            probs = torch.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


@torch.no_grad()
def estimate_loss(model: nn.Module, cfg: AttentionTrainConfig, model_cfg: AttentionModelConfig, eval_iters: int) -> Dict[str, float]:
    out = {}
    model.eval()
    for split in ("train", "val"):
        losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(split, cfg.batch_size, model_cfg.max_seq_len, device)
            _, loss = model(xb, yb)
            losses.append(float(loss.item()))
        out[split] = float(np.mean(losses))
    model.train()
    return out

In [10]:
# Smoke test all requested attention types
with torch.no_grad():
    x = torch.randn(2, 64, model_cfg.d_model, device=device)
    context = torch.randn(2, 80, model_cfg.d_model, device=device)

    attn_type_1 = MultiHeadSelfAttention(model_cfg.d_model, model_cfg.n_heads, dropout=0.0, use_rope=model_cfg.use_rope).to(device)
    out_1 = attn_type_1(x)

    attn_type_2 = CausalSelfAttention(model_cfg.d_model, model_cfg.n_heads, dropout=0.0, use_rope=model_cfg.use_rope).to(device)
    out_2 = attn_type_2(x)

    attn_type_3 = MultiQueryAttention(model_cfg.d_model, model_cfg.n_heads, dropout=0.0, causal=True, use_rope=model_cfg.use_rope).to(device)
    out_3 = attn_type_3(x)

    kv_heads = max(1, model_cfg.n_heads // 4)
    if model_cfg.n_heads % kv_heads != 0:
        kv_heads = 1
    attn_type_4 = GroupedQueryAttention(
        model_cfg.d_model, model_cfg.n_heads, n_kv_heads=kv_heads, dropout=0.0, causal=True, use_rope=model_cfg.use_rope
    ).to(device)
    out_4 = attn_type_4(x)

    cross_attn = CrossAttention(model_cfg.d_model, model_cfg.n_heads, n_kv_heads=kv_heads, dropout=0.0, use_rope=False).to(device)
    out_cross = cross_attn(x, context=context)

print("Type 1 MHA output:", tuple(out_1.shape))
print("Type 2 Masked Causal output:", tuple(out_2.shape))
print("Type 3 MQA output:", tuple(out_3.shape))
print("Type 4 GQA output:", tuple(out_4.shape))
print("Bonus Cross-Attention output:", tuple(out_cross.shape))

Type 1 MHA output: (2, 64, 192)
Type 2 Masked Causal output: (2, 64, 192)
Type 3 MQA output: (2, 64, 192)
Type 4 GQA output: (2, 64, 192)
Bonus Cross-Attention output: (2, 64, 192)


In [15]:
def _save_checkpoint(
    path: Path,
    stage_idx: int,
    stage_name: str,
    step_in_stage: int,
    global_step: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    history: Dict[str, List[float]],
    model_cfg: AttentionModelConfig,
    train_cfg: AttentionTrainConfig,
    ) -> None:
    torch.save(
        {
            "stage_idx": stage_idx,
            "stage_name": stage_name,
            "step_in_stage": step_in_stage,
            "global_step": global_step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict() if device.type == "cuda" else None,
            "history": history,
            "model_config": model_cfg.__dict__,
            "train_config": train_cfg.__dict__,
            "stage_plan": ATTENTION_STAGE_PLAN,
            "stage_step_scale": STAGE_STEP_SCALE,
        },
        path,
    )


resume_state = None
if RESUME_FROM_CHECKPOINT and CHECKPOINT_FILE.exists():
    resume_state = torch.load(CHECKPOINT_FILE, map_location=device)
    print("Resuming from checkpoint:", CHECKPOINT_FILE)

history = {"stage": [], "global_step": [], "train_loss": [], "val_loss": []}
global_step = 0
latest_model = None
latest_profile = ATTENTION_STAGE_PLAN[0]
start_stage_index = 0
start_step_in_stage = 0

if resume_state is not None:
    history = resume_state.get("history", history)
    global_step = int(resume_state.get("global_step", 0))
    start_stage_index = int(resume_state.get("stage_idx", 0))
    start_step_in_stage = int(resume_state.get("step_in_stage", 0))
    print(f"Resume state | stage_idx={start_stage_index}, step_in_stage={start_step_in_stage}, global_step={global_step}")

for stage_idx, stage_name in enumerate(ATTENTION_STAGE_PLAN):
    base_model_cfg, base_train_cfg = profiles[stage_name]
    model_cfg = AttentionModelConfig(**base_model_cfg.__dict__)
    train_cfg = AttentionTrainConfig(**base_train_cfg.__dict__)

    scale = STAGE_STEP_SCALE[min(stage_idx, len(STAGE_STEP_SCALE) - 1)] if STAGE_STEP_SCALE else 1.0
    train_cfg.steps = max(1, int(train_cfg.steps * scale))
    train_cfg.warmup_steps = min(train_cfg.warmup_steps, max(1, train_cfg.steps // 2))

    print(f"\n===== Stage {stage_idx + 1}/{len(ATTENTION_STAGE_PLAN)} | {stage_name} =====")
    print(f"Stage steps={train_cfg.steps}, warmup={train_cfg.warmup_steps}, batch={train_cfg.batch_size}, seq={model_cfg.max_seq_len}")

    stage_embedding = pretrained_token_embedding
    if latest_model is not None:
        stage_embedding = latest_model.token_embed.weight.detach().cpu()

    model = PowerfulAttentionLM(model_cfg, pretrained_embedding=stage_embedding).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    stage_start_step = 0
    if resume_state is not None and stage_idx == start_stage_index:
        model.load_state_dict(resume_state["model_state_dict"])
        optimizer.load_state_dict(resume_state["optimizer_state_dict"])
        if device.type == "cuda" and resume_state.get("scaler_state_dict") is not None:
            scaler.load_state_dict(resume_state["scaler_state_dict"])
        stage_start_step = start_step_in_stage
        print(f"Loaded checkpoint weights for stage {stage_name} at step {stage_start_step}")

    if stage_idx < start_stage_index:
        continue

    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {num_params / 1_000_000:.2f}M")

    model.train()
    optimizer.zero_grad(set_to_none=True)
    stage_start_time = time.perf_counter()

    for step in range(stage_start_step, train_cfg.steps):
        lr = cosine_lr(step, train_cfg.steps, train_cfg.warmup_steps, train_cfg.lr)
        for group in optimizer.param_groups:
            group["lr"] = lr

        if step % train_cfg.eval_interval == 0 or step == train_cfg.steps - 1:
            losses = estimate_loss(model, train_cfg, model_cfg, eval_iters=train_cfg.eval_iters)
            history["stage"].append(stage_name)
            history["global_step"].append(global_step)
            history["train_loss"].append(losses["train"])
            history["val_loss"].append(losses["val"])
            print(
                f"stage={stage_name} | step={step:4d}/{train_cfg.steps} | global={global_step:5d} | "
                f"lr={lr:.6f} | train_loss={losses['train']:.4f} | val_loss={losses['val']:.4f}"
            )

        xb, yb = get_batch("train", train_cfg.batch_size, model_cfg.max_seq_len, device)

        if device.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                _, loss = model(xb, yb)
                loss = loss / train_cfg.grad_accum_steps
            scaler.scale(loss).backward()
        else:
            _, loss = model(xb, yb)
            loss = loss / train_cfg.grad_accum_steps
            loss.backward()

        should_step = ((step + 1) % train_cfg.grad_accum_steps == 0) or (step == train_cfg.steps - 1)
        if should_step:
            if train_cfg.clip_grad > 0:
                if device.type == "cuda":
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.clip_grad)

            if device.type == "cuda":
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        global_step += 1

        if ((step + 1) % CHECKPOINT_EVERY == 0) or (step == train_cfg.steps - 1):
            _save_checkpoint(
                CHECKPOINT_FILE,
                stage_idx=stage_idx,
                stage_name=stage_name,
                step_in_stage=step + 1,
                global_step=global_step,
                model=model,
                optimizer=optimizer,
                scaler=scaler,
                history=history,
                model_cfg=model_cfg,
                train_cfg=train_cfg,
            )

    stage_elapsed = time.perf_counter() - stage_start_time
    print(f"Completed stage {stage_name} in {stage_elapsed:.2f}s")

    latest_model = model
    latest_profile = stage_name
    resume_state = None

if latest_model is None:
    raise RuntimeError("No model was trained. Check ATTENTION_STAGE_PLAN and checkpoint settings.")

model = latest_model
profile_name = latest_profile
elapsed = sum(0.0 for _ in [])  # placeholder for backward compatibility with downstream cell
print("Training curriculum completed. Final profile:", profile_name)


===== Stage 1/2 | cpu_safe =====
Stage steps=22, warmup=8, batch=24, seq=128
Trainable parameters: 2.52M
stage=cpu_safe | step=   0/22 | global=    0 | lr=0.000037 | train_loss=6.8397 | val_loss=6.8441
stage=cpu_safe | step=  15/22 | global=   15 | lr=0.000165 | train_loss=4.9874 | val_loss=5.0308
stage=cpu_safe | step=  21/22 | global=   21 | lr=0.000033 | train_loss=4.6831 | val_loss=4.7511
Completed stage cpu_safe in 23.25s

===== Stage 2/2 | cpu_quality =====
Stage steps=100, warmup=16, batch=16, seq=160
Trainable parameters: 5.82M
stage=cpu_quality | step=   0/100 | global=   22 | lr=0.000016 | train_loss=7.9484 | val_loss=7.9457
stage=cpu_quality | step=  20/100 | global=   42 | lr=0.000249 | train_loss=4.7485 | val_loss=4.8402
stage=cpu_quality | step=  40/100 | global=   62 | lr=0.000208 | train_loss=4.0758 | val_loss=4.2199
stage=cpu_quality | step=  60/100 | global=   82 | lr=0.000129 | train_loss=4.0479 | val_loss=4.1608
stage=cpu_quality | step=  80/100 | global=  102 | lr

In [16]:
if history["val_loss"]:
    best_val_loss = min(history["val_loss"])
    best_train_loss = min(history["train_loss"])
    val_ppl = math.exp(best_val_loss) if best_val_loss < 20 else float("inf")
    print(f"Best train loss: {best_train_loss:.4f}")
    print(f"Best val loss  : {best_val_loss:.4f}")
    print(f"Approx val ppl : {val_ppl:.2f}")

Best train loss: 3.9869
Best val loss  : 4.1137
Approx val ppl : 61.17


In [17]:
prompt = "Dorothy "
prompt_ids = tokenizer.encode(prompt, add_bos=True, add_eos=False)
prompt_tensor = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)

with torch.no_grad():
    generated = model.generate(
        prompt_tensor,
        max_new_tokens=train_cfg.max_new_tokens,
        temperature=0.9,
        top_k=60,
    )

generated_text = tokenizer.decode(generated[0].tolist(), skip_special_tokens=True)
print("Prompt:", repr(prompt))
print("Generated sample:")
print(generated_text[:1200])

artifact_path = Path("full_architecture_model_wizard.pt")
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_config": model_cfg.__dict__,
        "train_config": train_cfg.__dict__,
        "attention_profile": profile_name,
        "attention_stage_plan": ATTENTION_STAGE_PLAN,
        "attention_stage_step_scale": STAGE_STEP_SCALE,
        "checkpoint_file": str(CHECKPOINT_FILE),
        "embedding_profile": embedding_profile_name,
        "history": history,
        "tokenizer_json": str(tokenizer_path),
        "embedding_artifact": str(embedding_artifact_path),
    },
    artifact_path,
)
print("Saved full architecture artifact:", artifact_path.resolve())

Prompt: 'Dorothy '
Generated sample:
Dorothy the of is you upon in one her he from to and the his asked upon upon we at said he not and of Wizard and of to her and with to in I the of and of
the is and the you a we so said of and like as Wizard their there and , that to the the the of she but they the and and to with 
Saved full architecture artifact: D:\Desktop\Mini_Generative_Pretrained_Transformer\Research\full_architecture_model_wizard.pt


## Unified Pipeline Notes

- This notebook can train tokenizer, embeddings, and attention LM in one run.
- Full upstream retrain: set `FORCE_RETRAIN_TOKENIZER = True` and `FORCE_RETRAIN_EMBEDDINGS = True`.
- Small-to-large curriculum is controlled by `ATTENTION_STAGE_PLAN` and `STAGE_STEP_SCALE`.
- Checkpointing is enabled via `CHECKPOINT_FILE` and `CHECKPOINT_EVERY`, with resume by `RESUME_FROM_CHECKPOINT`.
- To scale on RTX 4060, use stage plans like `["rtx_4060_balanced", "rtx_4060_quality"]`.
- If memory is tight, reduce in this order: batch size -> sequence length -> layer count -> d_model.